# Openq2222AOA + CVAR: preliminary benchmarking

In [7]:
from openqaoa.problems.maximumcut import MaximumCut
from openqaoa.backends import create_device

import networkx as nx
from openqaoa import QAOA  
from openqaoa.problems import MaximumCut
from openqaoa.utilities import ground_state_hamiltonian
import matplotlib.pyplot as plt
from openqaoa.utilities import plot_graph
from qiskit_aer.noise import (NoiseModel, depolarizing_error)


In [16]:
import json
import os

## create noise model

In [8]:

one_qubit_gates = ['h','rx']
two_qubits_gates = ['rzz']

#create depol. noise
def add_depolarizing_error(noise_model,prob1, prob2):
    noise_model = add_one_qubit_depolarizing_error(noise_model,prob1)
    noise_model = add_two_qubits_depolarizing_error(noise_model,prob2)
    return noise_model

#create 1 qubit depol. noise
def add_one_qubit_depolarizing_error(noise_model,prob):
    error = depolarizing_error(prob, 1)
    noise_model.add_all_qubit_quantum_error(error,one_qubit_gates)
    return noise_model

#create 2 qubits depol.noise
def add_two_qubits_depolarizing_error(noise_model,prob):
    error = depolarizing_error(prob, 2)
    noise_model.add_all_qubit_quantum_error(error, two_qubits_gates)
    return noise_model

noise_model = add_depolarizing_error(NoiseModel(),0.0001989, 0.007905) #ibm_quebec, 19/01/2024

## create MaxCut Problems

In [11]:
graph1 = nx.Graph()
graph1.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph1.add_edges_from([(0, 6), (0, 8), (1, 5), (2, 7), (2, 8), (3, 5), (3, 7), (4, 8), (6, 7), (7, 9), (8, 9)])

graph2 = nx.Graph()
graph2.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph2.add_edges_from([(0, 2), (0, 3), (0, 4), (0, 5), (0, 7), (0, 8), (1, 4), (1, 6), (1, 8), (1, 9), (2, 4), (2, 6), (3, 4), (3, 6), (3, 8), (3, 9), (4, 5), (4, 7), (4, 9), (5, 8), (6, 7), (7, 8), (7, 9), (8, 9)])

graph3 = nx.Graph()
graph3.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph3.add_edges_from([(0, 1), (0, 2), (0, 4), (0, 9), (1, 5), (1, 6), (1, 7), (1, 8), (2, 3), (2, 4), (2, 6), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (4, 5), (4, 7), (4, 8), (5, 6), (5, 7), (5, 8), (5, 9), (6, 8), (6, 9), (7, 8), (7, 9)])

graph4 = nx.Graph()
graph4.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph4.add_edges_from([(0, 3), (1, 3), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (2, 3), (3, 4), (3, 6), (3, 7), (4, 5), (4, 6), (4, 9), (5, 6), (5, 7), (5, 8), (6, 7), (6, 8), (6, 9), (7, 9), (8, 9)])

graph5 = nx.Graph()
graph5.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph5.add_edges_from([(0, 5), (0, 7), (0, 8), (1, 2), (1, 3), (1, 4), (1, 6), (1, 9), (2, 3), (2, 4), (2, 5), (2, 8), (2, 9), (3, 5), (3, 6), (3, 7), (3, 9), (4, 6), (4, 7), (4, 9), (5, 6), (6, 7), (6, 8), (7, 9)])

mc1 = MaximumCut(graph1)
mc2 = MaximumCut(graph2)
mc3 = MaximumCut(graph3)
mc4 = MaximumCut(graph4)
mc5 = MaximumCut(graph5)

mcs = [mc1, mc2, mc3, mc4, mc5]

In [ ]:
ps = [1,2,3,4,5]
ps = [1,2,3,4,5]
param_types = ["extended", "standard"]
init_types = ["ramp", "rand"]
mixer_hams = ["x", "xy"]
optimizers = ["COBYLA", "Powell", "Nelder-Mead"]

alpha_values = [ 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] #look into [0,1]

for i,mc in enumerate(mcs):
    for alpha in alpha_values:
        for p in ps:
            for param_type in param_types:
                for init_type in init_types:
                    for mixer_hamiltonian in mixer_hams:
                        for optimizer in optimizers:
                            mc = mc.qubo

                            qiskit_device = create_device(location='local', name='qiskit.shot_simulator')

                            #noisy + CVAR
                            q = QAOA()
                            q.set_device(qiskit_device)
                            q.set_circuit_properties(p=p, param_type=param_type, init_type=init_type, mixer_hamiltonian=mixer_hamiltonian)
                            q.set_classical_optimizer(method=optimizer, maxiter=200, tol=0.001,
                                                    optimization_progress=True, cost_progress=True, parameter_log=True)
                            q.set_backend_properties(cvar_alpha = alpha, n_shots=5000, seed_simulator=1, noise_model=noise_model)
                            q.compile(mc)
                            q.optimize()
                            correct_solution = ground_state_hamiltonian(q.cost_hamil)
                            print(correct_solution)
                            opt_results = q.result

                            filename = 'prob%s/noisyCVAR/prob%s_p%s-%s-%s-%s-%s_noiseless.json'% (str(i),str(alpha),str(p), str(param_type), str(init_type), str(mixer_hamiltonian), str(optimizer))
                            os.makedirs(os.path.dirname(filename), exist_ok=True)

                            with open(filename, 'w') as archivo:
                                json.dump(opt_results.asdict(), archivo)

                            #noiseless + CVAR
                            q2 = QAOA()
                            q2.set_device(qiskit_device)
                            q2.set_circuit_properties(p=p, param_type=param_type, init_type=init_type, mixer_hamiltonian=mixer_hamiltonian)
                            q2.set_classical_optimizer(method=optimizer, maxiter=200, tol=0.001,
                                                    optimization_progress=True, cost_progress=True, parameter_log=True)
                            q2.set_backend_properties(cvar_alpha = alpha, n_shots=5000, seed_simulator=1, noise_model=noise_model)
                            q2.compile(mc)
                            q2.optimize()
                            correct_solution2 = ground_state_hamiltonian(q2.cost_hamil)
                            print(correct_solution2)
                            opt_results2 = q2.result
                            filename = 'prob%s/noiselessCVAR/prob%s_p%s-%s-%s-%s-%s_noiseless.json'% (str(i),str(alpha),str(p), str(param_type), str(init_type), str(mixer_hamiltonian), str(optimizer))
                            os.makedirs(os.path.dirname(filename), exist_ok=True)

                            with open(filename, 'w') as archivo:
                                json.dump(opt_results.asdict(), archivo)
                            #noisy + EXP
                            q3 = QAOA()
                            q3.set_device(qiskit_device)
                            q3.set_circuit_properties(p=p, param_type=param_type, init_type=init_type, mixer_hamiltonian=mixer_hamiltonian)
                            q3.set_classical_optimizer(method=optimizer, maxiter=200, tol=0.001,
                                                    optimization_progress=True, cost_progress=True, parameter_log=True)
                            q3.set_backend_properties(n_shots=5000, seed_simulator=1, noise_model=noise_model)
                            q3.compile(mc)
                            q3.optimize()
                            correct_solution = ground_state_hamiltonian(q3.cost_hamil)
                            print(correct_solution)
                            opt_results3 = q3.result
                            filename = 'prob%s/noisyEXP/prob%s_p%s-%s-%s-%s-%s_noiseless.json'% (str(i),str(alpha),str(p), str(param_type), str(init_type), str(mixer_hamiltonian), str(optimizer))
                            os.makedirs(os.path.dirname(filename), exist_ok=True)

                            with open(filename, 'w') as archivo:
                                json.dump(opt_results.asdict(), archivo)
                            #noiseless + EXP
                            q4 = QAOA()
                            q4.set_device(qiskit_device)
                            q4.set_circuit_properties(p=p, param_type=param_type, init_type=init_type, mixer_hamiltonian=mixer_hamiltonian)
                            q4.set_classical_optimizer(method=optimizer, maxiter=200, tol=0.001,
                                                    optimization_progress=True, cost_progress=True, parameter_log=True)
                            q4.set_backend_properties(n_shots=5000, seed_simulator=1, noise_model=noise_model)
                            q4.compile(mc)
                            q4.optimize()
                            correct_solution2 = ground_state_hamiltonian(q4.cost_hamil)
                            print(correct_solution2)
                            opt_results4 = q4.result
                            filename = 'prob%s/noiselessEXP/prob%s_p%s-%s-%s-%s-%s_noiseless.json'% (str(i),str(alpha),str(p), str(param_type), str(init_type), str(mixer_hamiltonian), str(optimizer))
                            os.makedirs(os.path.dirname(filename), exist_ok=True)

                            with open(filename, 'w') as archivo:
                                json.dump(opt_results.asdict(), archivo)

                            

/home/marcovenere/qosf_venv_2/lib/python3.10/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(


(-9.0, ['0000010110', '1000010110', '0000011110', '1111100001', '0111101001', '1111101001'])
